# Notebook 02: Exploratory Data Analysis**Goal:** Visualize the data, find patterns, correlations, and tell the story.Key questions:1. What drives price most strongly?2. How does location affect pricing?3. Is age actually important?4. Are there non-linear relationships?

In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsimport syssys.path.insert(0, "../src")from housing.data.load_data import load_raw_datafrom housing.visualization.visualize import (    plot_distributions, plot_correlation_matrix, plot_price_by_location)plt.style.use("seaborn-v0_8-whitegrid")sns.set_palette("husl")# Load datadf = load_raw_data("../data/raw/housing.csv")

## 1. Distribution Overview

In [ ]:
fig = plot_distributions(df, save_path="../reports/figures/eda_distributions.png")plt.show()

**Observations:**- Price is roughly right-skewed but not extreme- Square footage spans 701–3499 (good range)- Age is fairly uniform 1–49 years

## 2. Correlation Analysis

In [ ]:
fig = plot_correlation_matrix(df, save_path="../reports/figures/eda_correlations.png")plt.show()

**Key Correlations with Price:**- square_feet: **0.80** ← Dominant driver- bedrooms: 0.13- bathrooms: 0.12- age: **-0.09** ← Surprisingly weak!**Multicollinearity alert:** bedrooms and bathrooms are 0.93 correlated.

## 3. Location Analysis

In [ ]:
fig = plot_price_by_location(df, save_path="../reports/figures/eda_location.png")plt.show()

**Location creates three distinct markets:**| Location | Median Price | Price/SqFt ||----------|-------------|------------|| Downtown | $504,100 | $243 || Suburbs  | $374,000 | $175 || Rural    | $284,200 | $136 |Downtown commands an **80% premium** per square foot over Rural.

## 4. Age Deep DiveThe correlation was weak (-0.09). Let's bin it and see if there's a non-linear pattern.

In [ ]:
df['age_bin'] = pd.cut(df['age'], bins=[0, 10, 20, 30, 40, 50],                         labels=['0-10', '11-20', '21-30', '31-40', '41-50'])age_price = df.groupby('age_bin')['price'].mean()fig, ax = plt.subplots(figsize=(8, 5))age_price.plot(kind='bar', ax=ax, color='teal', edgecolor='white')ax.set_title('Average Price by Age Group')ax.set_xlabel('Age Group (years)')ax.set_ylabel('Average Price ($)')ax.tick_params(axis='x', rotation=0)plt.tight_layout()plt.savefig("../reports/figures/eda_age_bins.png", dpi=150)plt.show()

**Finding:** Prices are nearly flat across all age groups. A 40-year-old house is not systematically cheaper than a 5-year-old one.**Implication:** Age is not a strong predictor in this market. Location and size completely swamp its effect.

## 5. Size vs Price by LocationDoes the relationship between square footage and price vary by location?

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))for loc in df['location'].unique():    subset = df[df['location'] == loc]    ax.scatter(subset['square_feet'], subset['price'], label=loc, alpha=0.6, s=40)ax.set_xlabel('Square Feet')ax.set_ylabel('Price ($)')ax.set_title('Square Feet vs Price by Location')ax.legend()plt.tight_layout()plt.savefig("../reports/figures/eda_size_location.png", dpi=150)plt.show()

**Finding:** All three locations show a strong positive linear relationship between square footage and price, but with different intercepts (Downtown highest).This suggests an **interaction effect** — location modifies the value of square footage.Tree-based models will capture this naturally.

## EDA Summary| Insight | Action for Modeling ||---------|-------------------|| Square footage dominates (r=0.80) | Primary feature || Location creates 3 distinct markets | One-hot encode, expect strong effect || Age is basically irrelevant | Include but don't expect much || Bedrooms/bathrooms collinear (r=0.93) | Create `total_rooms` to reduce redundancy || Non-linear interactions exist | Tree models > Linear for capturing interactions |